# FC PCA

Run matched no-SES and SES-residualized pMTG functional-connectivity PCA workflows. Revised cognitive EFA factor scores are loaded from `PCA_tasks.ipynb` exports so task EFA is not refit here.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.decomposition import PCA
from scipy import stats
from sklearn.linear_model import LinearRegression
from statsmodels.stats.multitest import multipletests
import seaborn as sns
from matplotlib.patches import Patch

sns.set_style('whitegrid')
sns.set_palette('colorblind')
palette = sns.color_palette('colorblind')
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 26,
    'axes.labelsize': 22,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
})

network_colors = {
    'DMN': '#fb2e2e',
    'VAN': '#40cce9',
    'LANG': '#40cce9',
    'TPOLE': '#2e87ae',
    'AUD': '#ce8fff',
    'AMN': '#8a2edb',
    'DAN': '#2efe2e',
    'FP': '#ffff2e',
    'MTL': '#89fd89',
    'PMN': '#2e2eff',
    'PON': '#ebebeb',
    'SAL': '#2e2e2e',
    'SMd': '#8fffff',
    'SMl': '#ffa12e',
    'VIS': '#2e2eb3',
}


In [ ]:
INVALID_CODES = (555, 777, 888, 999)

STANDARD_FC_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]

DEFAULT_CATEGORICAL_COVARIATES = {
    'demo_sex_v2',
    'site_id_l',
    'ehi1b',
}


def standardize_subject_id(subject_ids):
    return (
        subject_ids.astype(str)
        .str.replace('_', '', regex=False)
        .str.replace('^sub-', '', regex=True)
    )


def load_motion_qa(motion_qa_path, motion_column='mean_fd_0.20'):
    motion_qa = pd.read_csv(motion_qa_path)
    motion_qa = motion_qa[['src_subject_id', motion_column]].copy()
    motion_qa['src_subject_id'] = standardize_subject_id(motion_qa['src_subject_id'])
    motion_qa = motion_qa.drop_duplicates(subset=['src_subject_id'])
    motion_qa[motion_column] = pd.to_numeric(motion_qa[motion_column], errors='coerce')
    return motion_qa


def merge_motion_qa(df, motion_qa_path, motion_column='mean_fd_0.20', how='left'):
    motion_qa = load_motion_qa(motion_qa_path, motion_column=motion_column)

    merged = df.copy()
    merged['src_subject_id'] = standardize_subject_id(merged['src_subject_id'])
    if motion_column in merged.columns:
        existing_motion = merged.groupby('src_subject_id')[motion_column].first()
        merged = merged.drop(columns=[motion_column])
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
        merged[motion_column] = merged[motion_column].fillna(
            merged['src_subject_id'].map(existing_motion)
        )
    else:
        merged = merged.merge(motion_qa, on='src_subject_id', how=how)
    return merged


def complete_covariate_mask(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    categorical_covariates = set(categorical_covariates)
    complete = pd.Series(True, index=df.index)

    for covariate in covariates:
        is_categorical = df[covariate].dtype == 'object' or covariate in categorical_covariates
        if is_categorical:
            complete &= df[covariate].notna()
        else:
            complete &= pd.to_numeric(df[covariate], errors='coerce').notna()

    return complete


def encode_regression_covariates(df, covariates, categorical_covariates=DEFAULT_CATEGORICAL_COVARIATES):
    encoded_parts = []
    categorical_covariates = set(categorical_covariates)

    for covariate in covariates:
        if df[covariate].dtype == 'object' or covariate in categorical_covariates:
            encoded_parts.append(
                pd.get_dummies(
                    df[covariate],
                    prefix=covariate,
                    drop_first=True,
                    dtype=float,
                )
            )
        else:
            encoded_parts.append(
                pd.to_numeric(df[covariate], errors='coerce').to_frame(covariate)
            )

    if not encoded_parts:
        return pd.DataFrame(index=df.index)
    return pd.concat(encoded_parts, axis=1)


def residualize_fc_profiles(df, fc_columns, covariates=STANDARD_FC_COVARIATES):
    df = df.copy()
    covariate_complete = complete_covariate_mask(df, covariates)

    validity_groups = {}
    for column in fc_columns:
        valid_idx = df[column].notnull() & covariate_complete
        key = valid_idx.to_numpy(dtype=np.bool_).tobytes()
        if key not in validity_groups:
            validity_groups[key] = (valid_idx, [])
        validity_groups[key][1].append(column)

    residual_frames = []
    for valid_idx, columns in validity_groups.values():
        output_columns = [column + '_resid' for column in columns]
        residuals = pd.DataFrame(np.nan, index=df.index, columns=output_columns)
        if valid_idx.sum() > 0:
            observed = df.loc[valid_idx, columns].to_numpy()
            covariate_matrix = encode_regression_covariates(
                df.loc[valid_idx],
                covariates,
            )
            if covariate_matrix.shape[1] == 0:
                predicted = np.tile(observed.mean(axis=0), (len(observed), 1))
            else:
                model = LinearRegression()
                model.fit(covariate_matrix, observed)
                predicted = model.predict(covariate_matrix)
            residuals.loc[valid_idx, output_columns] = observed - predicted
        residual_frames.append(residuals)

    if residual_frames:
        df = pd.concat([df, *residual_frames], axis=1)

    return df


def compute_correlations(df, measures, fc_columns, required_nonmissing=None):
    results = []
    required_nonmissing = list(required_nonmissing or [])

    for measure in measures:
        for column in fc_columns:
            analysis_columns = list(dict.fromkeys([column, measure, *required_nonmissing]))
            temp_df = df[analysis_columns].dropna()

            if (
                len(temp_df) < 2
                or temp_df[column].nunique(dropna=True) < 2
                or temp_df[measure].nunique(dropna=True) < 2
            ):
                r_value = np.nan
                p_value = np.nan
            else:
                r_value, p_value = stats.pearsonr(temp_df[column], temp_df[measure])

            results.append({
                'measure': measure,
                'col': column,
                'r': r_value,
                'p': p_value,
                'n': len(temp_df),
            })

    return pd.DataFrame(results)


def apply_multiple_comparison_corrections(results_df, alpha=0.05, fdr_group_col=None):
    if results_df.empty:
        return results_df.assign(
            p_bonf=pd.Series(dtype=float),
            sig_bonf=pd.Series(dtype=bool),
            p_fdr=pd.Series(dtype=float),
            sig_fdr=pd.Series(dtype=bool),
        )

    corrected = results_df.copy()
    n_tests = len(corrected)
    corrected['p_bonf'] = np.minimum(corrected['p'] * n_tests, 1.0)
    corrected['sig_bonf'] = corrected['p_bonf'] < alpha

    valid_p = corrected['p'].notna()
    corrected['p_fdr'] = np.nan
    corrected['sig_fdr'] = False

    if fdr_group_col is None:
        fdr_families = pd.Series('all', index=corrected.index)
    else:
        fdr_families = corrected[fdr_group_col].astype('string').fillna('<missing>')

    for family in fdr_families.unique():
        family_valid_p = valid_p & fdr_families.eq(family)
        if family_valid_p.any():
            reject, p_fdr, _, _ = multipletests(
                corrected.loc[family_valid_p, 'p'],
                alpha=alpha,
                method='fdr_bh',
            )
            corrected.loc[family_valid_p, 'p_fdr'] = p_fdr
            corrected.loc[family_valid_p, 'sig_fdr'] = reject

    return corrected


In [ ]:
DATA_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/wrangled_pMTG_FC_data_midb61_meanFC.csv'
MOTION_QA_PATH = '/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/motion_QA_results.csv'
PCA_RESULTS_DIR = Path('/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/pca_results')
PCA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df = merge_motion_qa(df, MOTION_QA_PATH, how='left')
missing_motion = df['mean_fd_0.20'].isna().sum() if 'mean_fd_0.20' in df.columns else len(df)
print(f'Loaded {len(df)} rows from {DATA_PATH}')
print(f'Missing mean_fd_0.20 values after motion QA merge: {missing_motion}')


## FC Analysis Settings

Run `PCA_tasks.ipynb` first so the revised cognitive EFA factor-score exports exist. The SES model residualizes raw FC columns once with INR included alongside the standard FC covariates.


In [ ]:
SES_VARIABLE = 'inr'
SES_RESIDUALIZATION_COVARIATE = 'inr'
ALPHA = 0.05
N_FC_BOOTSTRAPS = 50000

# Only Fisher-z FC features are allowed in FC PCA; generated _full networks are dropped.
full_fc_cols = [column for column in df.columns if '_fz' in str(column) and '_full' in str(column)]
if full_fc_cols:
    df = df.drop(columns=full_fc_cols)
print(f'Dropped {len(full_fc_cols)} generated full-network Fisher-z FC columns from the FC PCA dataframe.')

raw_fc_cols = [col for col in df.columns if col.endswith('_fz') and '_full' not in col]

required_inr_cols = [SES_VARIABLE, 'inr_missing', 'poverty_line_2017']
missing_inr_cols = [col for col in required_inr_cols if col not in df.columns]

print(f'Missing raw INR values excluded from FC-behavior correlations: {df[SES_VARIABLE].isna().sum()}')

FC_PCA_COVARIATES = [
    'demo_sex_v2',
    'interview_age',
    'site_id_l',
    'ehi1b',
    'mean_fd_0.20',
]

PCA_MODEL_SPECS = [
    {
        'model_name': 'without_ses_residualization',
        'label': 'No SES residualization',
        'short_label': 'No SES',
        'prefix': 'no_ses',
        'fc_covariates': FC_PCA_COVARIATES,
    },
    {
        'model_name': 'with_ses_residualization',
        'label': 'SES residualization',
        'short_label': 'SES',
        'prefix': 'ses',
        'fc_covariates': FC_PCA_COVARIATES + [SES_RESIDUALIZATION_COVARIATE],
    },
]

for spec in PCA_MODEL_SPECS:
    missing_fc_covariates = [
        covariate for covariate in spec['fc_covariates'] if covariate not in df.columns
    ]

    scores_path = PCA_RESULTS_DIR / f'efa_cognitive_scores_{spec["model_name"]}.csv'

print(f'Raw FC columns available for PCA/correlations: {len(raw_fc_cols)}')
print('FC PCA model specs:')
for spec in PCA_MODEL_SPECS:
    print(f"- {spec['model_name']}")
    print(f"  FC covariates: {spec['fc_covariates']}")


## Shared FC PCA Helpers


In [ ]:
def residualize_fc_variant(data, raw_columns, covariates, output_suffix):
    """Residualize raw FC columns once using the full covariate set."""
    helper_output_cols = [f'{col}_resid' for col in raw_columns]
    output_cols = [f'{col}_{output_suffix}_resid' for col in raw_columns]

    helper_input = data.drop(columns=helper_output_cols, errors='ignore')
    helper_output = residualize_fc_profiles(
        helper_input,
        raw_columns,
        covariates=covariates,
    )

    residualized = data.copy()
    for helper_col, output_col in zip(helper_output_cols, output_cols):
        residualized[output_col] = pd.to_numeric(helper_output[helper_col], errors='coerce')
    return residualized, output_cols


def attach_score_columns(data, scores):
    updated = data.copy()
    for score_col in scores.columns:
        updated[score_col] = scores[score_col]
    return updated


def load_cognitive_efa_scores(model_name):
    # Load the revised exploratory latent factors exported by PCA_tasks.ipynb.
    scores_path = PCA_RESULTS_DIR / f'efa_cognitive_scores_{model_name}.csv'
    cognitive_scores = pd.read_csv(scores_path)
    score_cols = [col for col in cognitive_scores.columns if col != 'src_subject_id' and '_EF' in col]
    return cognitive_scores, score_cols


def strip_fc_resid_suffix(column):
    text = str(column)
    for suffix in ('_no_ses_resid', '_ses_resid', '_resid'):
        if text.endswith(suffix):
            return text[: -len(suffix)]
    return text


def parsed_fc_parts(column):
    parts = strip_fc_resid_suffix(column).split('_')
    return parts[0], parts[1]


def plot_label_for(base_net, net_hemi):
    base = base_net.upper()
    hemi = net_hemi.lower()

    if base == 'CO':
        base = 'AMN'
    if base == 'SMD':
        base = 'SMd'
    if base == 'SML':
        base = 'SMl'
    if base == 'TPOLE':
        base = 'TPOLE'

    if base == 'VAN' and hemi == 'left':
        return 'LANG'
    if base == 'VAN' and hemi == 'right':
        return 'VAN'

    return base


def build_fc_plot_groups(feature_cols):
    base_nets_present = sorted({parsed_fc_parts(col)[0].upper() for col in feature_cols})

    lr_plot_groups = []
    for base in base_nets_present:
        if base == 'VAN':
            lr_plot_groups.append((base, 'left', 'LANG'))
            lr_plot_groups.append((base, 'right', 'VAN'))
        else:
            lr_plot_groups.append((base, 'left', plot_label_for(base, 'left')))
            lr_plot_groups.append((base, 'right', plot_label_for(base, 'right')))

    mean_plot_groups = []
    for base in base_nets_present:
        if base == 'VAN':
            mean_plot_groups.append((base, 'left_avg', 'LANG'))
            mean_plot_groups.append((base, 'right_avg', 'VAN'))
        else:
            mean_plot_groups.append((base, 'avg', plot_label_for(base, 'left')))

    lr_xlabels = []
    for _, hemi, label in lr_plot_groups:
        if label in ['LANG', 'VAN']:
            lr_xlabels.append(label)
        else:
            lr_xlabels.append(f'{label}\n{hemi}')

    mean_xlabels = [label for _, _, label in mean_plot_groups]
    return lr_plot_groups, mean_plot_groups, lr_xlabels, mean_xlabels


def pick_fc_index(feature_cols, base_net, net_hemi, pmtg_hemi):
    token = f'_{pmtg_hemi}_'
    matches = [
        i for i, col in enumerate(feature_cols)
        if parsed_fc_parts(col)[0].upper() == base_net.upper()
        and parsed_fc_parts(col)[1].lower() == net_hemi.lower()
        and token in strip_fc_resid_suffix(col)
    ]
    return matches[0]


def run_fc_pca_with_bootstrap(data, feature_cols, analysis_label, score_prefix, n_boot=N_FC_BOOTSTRAPS):
    print(f'Found {len(feature_cols)} {analysis_label} FC features for PCA.')

    fc_input = data[feature_cols].apply(pd.to_numeric, errors='coerce')
    fc_complete = fc_input.dropna().copy()
    n_missing_rows = len(fc_input) - len(fc_complete)
    print(f'{analysis_label} complete FC rows for PCA: {len(fc_complete)}/{len(fc_input)}')
    if n_missing_rows:
        print(f'{analysis_label} rows excluded from FC PCA for missing FC/covariate data: {n_missing_rows}')

    fc_centered = fc_complete - fc_complete.mean()
    x_matrix = fc_centered.to_numpy()
    n_features = x_matrix.shape[1]

    pca = PCA(random_state=42)
    scores_array = pca.fit_transform(x_matrix)
    reference_components = pca.components_
    n_components = reference_components.shape[0]

    loadings = (pca.components_.T * np.sqrt(pca.explained_variance_)).T

    boot_loadings = np.zeros((n_boot, n_components, n_features))
    boot_var_exp = np.zeros((n_boot, n_components))

    for boot_idx in range(n_boot):
        sample_idx = np.random.choice(x_matrix.shape[0], size=x_matrix.shape[0], replace=True)
        x_sample = x_matrix[sample_idx, :]

        boot_pca = PCA(random_state=42)
        boot_pca.fit(x_sample)
        boot_components = boot_pca.components_
        boot_var_exp[boot_idx, :] = boot_pca.explained_variance_ratio_

        boot_true = (boot_components.T * np.sqrt(boot_pca.explained_variance_)).T
        for pc_idx in range(n_components):
            if np.corrcoef(reference_components[pc_idx], boot_components[pc_idx])[0, 1] < 0:
                boot_true[pc_idx] *= -1

        boot_loadings[boot_idx, :, :] = boot_true

    ci_lower = np.percentile(boot_loadings, 2.5, axis=0)
    ci_upper = np.percentile(boot_loadings, 97.5, axis=0)
    var_exp_std = np.std(boot_var_exp, axis=0)

    score_cols = [f'{score_prefix}_PC{i + 1}' for i in range(scores_array.shape[1])]
    scores = pd.DataFrame(np.nan, index=fc_input.index, columns=score_cols, dtype=float)
    scores.loc[fc_complete.index, score_cols] = scores_array

    result = {
        'input': fc_input,
        'complete_input': fc_complete,
        'centered_input': fc_centered,
        'X': x_matrix,
        'pca': pca,
        'score_cols': score_cols,
        'scores': scores,
        'loadings': loadings,
        'boot_loadings': boot_loadings,
        'boot_var_exp': boot_var_exp,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'var_exp': pca.explained_variance_ratio_,
        'var_exp_std': var_exp_std,
    }

    print('\nVariance explained per PC:')
    for pc_idx in range(min(10, n_components)):
        print(
            f"PC{pc_idx + 1}: {result['var_exp'][pc_idx] * 100:.2f}% "
            f"(± {var_exp_std[pc_idx] * 100:.2f}%)"
        )

    plot_fc_loading_summaries(feature_cols, analysis_label, result)
    plot_fc_cumulative_variance(result, f'{analysis_label} Cumulative Variance Explained')
    print_fc_variance_table(result, f'{analysis_label} variance explained by each component')

    return result


def plot_fc_loading_summaries(feature_cols, analysis_label, result):
    loadings = result['loadings']
    boot_loadings = result['boot_loadings']
    ci_lower = result['ci_lower']
    ci_upper = result['ci_upper']
    n_pcs = loadings.shape[0]

    left_features = [i for i, col in enumerate(feature_cols) if '_L_' in strip_fc_resid_suffix(col)]
    right_features = [i for i, col in enumerate(feature_cols) if '_R_' in strip_fc_resid_suffix(col)]

    print(f'\nIdentified {len(left_features)} left pMTG')
    print(f'Identified {len(right_features)} right pMTG')
    print('\n===== OUTPUT SEPARATED BY _L_ AND _R_ FEATURES =====')

    lr_plot_groups, mean_plot_groups, lr_xlabels, mean_xlabels = build_fc_plot_groups(feature_cols)

    for pc_idx in range(min(10, n_pcs)):
        print(f'\n=========== {analysis_label} PC{pc_idx + 1} ===========\n')

        print('Top left pMTG features:')
        left_load = np.abs(loadings[pc_idx, left_features])
        left_sorted = np.argsort(left_load)[-12:][::-1]
        for idx in left_sorted:
            feature_idx = left_features[idx]
            feature = feature_cols[feature_idx]
            mean_loading = loadings[pc_idx, feature_idx]
            lower = ci_lower[pc_idx, feature_idx]
            upper = ci_upper[pc_idx, feature_idx]
            print(f'  {feature}: {mean_loading:.4f} [{lower:.4f}, {upper:.4f}]')

        print('\nTop right pMTG features:')
        right_load = np.abs(loadings[pc_idx, right_features])
        right_sorted = np.argsort(right_load)[-12:][::-1]
        for idx in right_sorted:
            feature_idx = right_features[idx]
            feature = feature_cols[feature_idx]
            mean_loading = loadings[pc_idx, feature_idx]
            lower = ci_lower[pc_idx, feature_idx]
            upper = ci_upper[pc_idx, feature_idx]
            print(f'  {feature}: {mean_loading:.4f} [{lower:.4f}, {upper:.4f}]')

        plot_fc_loading_bars(
            feature_cols,
            analysis_label,
            pc_idx,
            loadings,
            boot_loadings,
            lr_plot_groups,
            lr_xlabels,
            mode='network_hemisphere',
        )
        plot_fc_loading_bars(
            feature_cols,
            analysis_label,
            pc_idx,
            loadings,
            boot_loadings,
            mean_plot_groups,
            mean_xlabels,
            mode='network_average',
        )


def plot_fc_loading_bars(feature_cols, analysis_label, pc_idx, loadings, boot_loadings, plot_groups, xlabels, mode):
    x_values = np.arange(len(plot_groups))
    width = 0.38
    pmtg_left_means = []
    pmtg_right_means = []
    pmtg_left_err_lo = []
    pmtg_left_err_hi = []
    pmtg_right_err_lo = []
    pmtg_right_err_hi = []

    for base_net, group_mode, label in plot_groups:
        if group_mode == 'avg':
            left_indices = [
                pick_fc_index(feature_cols, base_net, 'left', 'L'),
                pick_fc_index(feature_cols, base_net, 'right', 'L'),
            ]
            right_indices = [
                pick_fc_index(feature_cols, base_net, 'left', 'R'),
                pick_fc_index(feature_cols, base_net, 'right', 'R'),
            ]
            left_boot = boot_loadings[:, pc_idx, left_indices].mean(axis=1)
            right_boot = boot_loadings[:, pc_idx, right_indices].mean(axis=1)
            left_mean = loadings[pc_idx, left_indices].mean()
            right_mean = loadings[pc_idx, right_indices].mean()
        elif group_mode == 'left_avg':
            left_indices = [pick_fc_index(feature_cols, base_net, 'left', 'L')]
            right_indices = [pick_fc_index(feature_cols, base_net, 'left', 'R')]
            left_boot = boot_loadings[:, pc_idx, left_indices[0]]
            right_boot = boot_loadings[:, pc_idx, right_indices[0]]
            left_mean = loadings[pc_idx, left_indices[0]]
            right_mean = loadings[pc_idx, right_indices[0]]
        elif group_mode == 'right_avg':
            left_indices = [pick_fc_index(feature_cols, base_net, 'right', 'L')]
            right_indices = [pick_fc_index(feature_cols, base_net, 'right', 'R')]
            left_boot = boot_loadings[:, pc_idx, left_indices[0]]
            right_boot = boot_loadings[:, pc_idx, right_indices[0]]
            left_mean = loadings[pc_idx, left_indices[0]]
            right_mean = loadings[pc_idx, right_indices[0]]
        else:
            left_idx = pick_fc_index(feature_cols, base_net, group_mode, 'L')
            right_idx = pick_fc_index(feature_cols, base_net, group_mode, 'R')
            left_boot = boot_loadings[:, pc_idx, left_idx]
            right_boot = boot_loadings[:, pc_idx, right_idx]
            left_mean = loadings[pc_idx, left_idx]
            right_mean = loadings[pc_idx, right_idx]

        left_lo, left_hi = np.percentile(left_boot, [2.5, 97.5])
        right_lo, right_hi = np.percentile(right_boot, [2.5, 97.5])

        pmtg_left_means.append(left_mean)
        pmtg_left_err_lo.append(max(left_mean - left_lo, 0))
        pmtg_left_err_hi.append(max(left_hi - left_mean, 0))
        pmtg_right_means.append(right_mean)
        pmtg_right_err_lo.append(max(right_mean - right_lo, 0))
        pmtg_right_err_hi.append(max(right_hi - right_mean, 0))

    figsize = (18, 6) if mode == 'network_hemisphere' else (16, 6)
    plt.figure(figsize=figsize)
    plt.bar(
        x_values - width / 2,
        pmtg_left_means,
        width,
        yerr=[pmtg_left_err_lo, pmtg_left_err_hi],
        capsize=4,
        label='Left pMTG',
        color=[network_colors.get(label, 'steelblue') for _, _, label in plot_groups],
        hatch='\\',
        edgecolor='black',
        linewidth=0.8,
    )
    plt.bar(
        x_values + width / 2,
        pmtg_right_means,
        width,
        yerr=[pmtg_right_err_lo, pmtg_right_err_hi],
        capsize=4,
        label='Right pMTG',
        color=[network_colors.get(label, 'steelblue') for _, _, label in plot_groups],
        hatch='//',
        edgecolor='black',
        linewidth=0.8,
    )

    plt.xticks(x_values, xlabels, rotation=45, ha='right', fontsize=12 if mode == 'network_hemisphere' else None)
    plt.title(f'{analysis_label} PC{pc_idx + 1} Loadings')
    plt.ylabel('Correlation with PC')
    plt.xlabel('Network hemisphere' if mode == 'network_hemisphere' else 'Network')
    plt.axhline(0, color='black', linewidth=0.8)

    handles = [
        Patch(facecolor='white', edgecolor='black', hatch='\\', label='Left pMTG', linewidth=1.0),
        Patch(facecolor='white', edgecolor='black', hatch='//', label='Right pMTG', linewidth=1.0),
    ]
    legend = plt.legend(handles=handles, frameon=True)
    legend.get_frame().set_facecolor('white')
    legend.get_frame().set_edgecolor('black')

    plt.tight_layout()
    plt.show()
    plt.close()


def plot_fc_cumulative_variance(fc_result, title):
    plt.figure(figsize=(18, 10))
    plt.plot(np.cumsum(fc_result['pca'].explained_variance_ratio_), marker='o', color='black')
    plt.title(title)
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.grid(True)
    plt.xticks(np.arange(0, 25, 1))
    plt.yticks(np.arange(0, 1.1, 0.1))
    xticks = plt.xticks()[0]
    xtick_labels = [str(int(x) + 1) for x in xticks]
    plt.xticks(xticks, xtick_labels)
    plt.xlim(0, 24)
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()
    plt.close()


def print_fc_variance_table(fc_result, title):
    print(title + ':')
    for idx, variance in enumerate(fc_result['pca'].explained_variance_ratio_):
        print(f'Component {idx + 1}: {variance:.4f} ({variance * 100:.2f}%)')
    print('Sum of first 10 components:', np.sum(fc_result['pca'].explained_variance_ratio_[:10]))


def compute_corrected_correlations(data, measures, targets, model_name, correlation_family, required_nonmissing=None):
    """Compute correlations and correct FDR within one model/family only."""
    results = compute_correlations(
        data,
        measures,
        targets,
        required_nonmissing=required_nonmissing,
    )
    results = apply_multiple_comparison_corrections(results, alpha=ALPHA)
    results['ses_model'] = model_name
    results['correlation_family'] = correlation_family
    results['cognitive_efa_factor'] = results['measure']
    results['target'] = results['col']
    results['abs_r'] = results['r'].abs()
    results['n_tests_in_fdr_family'] = len(results)
    return results.sort_values(['p', 'abs_r'], ascending=[True, False]).reset_index(drop=True)


def display_correlation_summary(results, title, top_n=20):
    print(title)
    print(f'Total tests in FDR family: {len(results)}')
    print(f'Sample-size range: {results["n"].min()}-{results["n"].max()}')

    print('\nBonferroni-significant correlations:')
    bonf = results[results['sig_bonf']].copy()
    if bonf.empty:
        print('None')
    else:
        display(bonf)

    print('\nFDR-significant correlations:')
    fdr = results[results['sig_fdr']].copy()
    if fdr.empty:
        print('None')
    else:
        display(fdr)

    print(f'\nTop {top_n} correlations by absolute r-value:')
    display(
        results.sort_values('abs_r', ascending=False).head(top_n)[
            ['cognitive_efa_factor', 'target', 'r', 'p', 'p_fdr', 'p_bonf', 'n', 'abs_r']
        ]
    )

def plot_residual_normality_diagnostics(data, residual_cols, labels=None, title_prefix='Residual normality', page_size=8, bins=30):
    """Display histogram and Q-Q plots for residual columns, plus summary moments."""
    residual_data = data[residual_cols].apply(pd.to_numeric, errors='coerce')
    labels = labels or residual_cols

    summary_rows = []
    for col, label in zip(residual_cols, labels):
        values = residual_data[col].dropna()
        summary_rows.append({
            'residual': col,
            'label': label,
            'n': len(values),
            'mean': values.mean(),
            'sd': values.std(ddof=1),
            'skew': stats.skew(values) if len(values) >= 3 else np.nan,
            'excess_kurtosis': stats.kurtosis(values) if len(values) >= 4 else np.nan,
        })

    summary = pd.DataFrame(summary_rows)
    print(f'{title_prefix}: residual normality summary')
    display(summary)

    for start in range(0, len(residual_cols), page_size):
        page_cols = residual_cols[start:start + page_size]
        page_labels = labels[start:start + page_size]
        fig, axes = plt.subplots(len(page_cols), 2, figsize=(13, 3.2 * len(page_cols)))
        axes = np.asarray(axes).reshape(len(page_cols), 2)

        for row_idx, (col, label) in enumerate(zip(page_cols, page_labels)):
            values = residual_data[col].dropna()
            hist_ax, qq_ax = axes[row_idx]

            if values.empty:
                hist_ax.text(0.5, 0.5, 'No non-missing values', ha='center', va='center')
                qq_ax.text(0.5, 0.5, 'No non-missing values', ha='center', va='center')
            else:
                sns.histplot(values, bins=bins, kde=True, ax=hist_ax, color=palette[0])
                hist_ax.axvline(values.mean(), color='black', linewidth=1.0, linestyle='--')
                hist_ax.set_xlabel('Residual')
                hist_ax.set_ylabel('Count')

                if len(values) >= 2:
                    stats.probplot(values, dist='norm', plot=qq_ax)
                    qq_ax.get_lines()[0].set_markerfacecolor(palette[1])
                    qq_ax.get_lines()[0].set_markeredgecolor(palette[1])
                    qq_ax.get_lines()[1].set_color('black')
                    qq_ax.get_lines()[1].set_linewidth(1.0)
                else:
                    qq_ax.text(0.5, 0.5, 'Need at least 2 values', ha='center', va='center')

            hist_ax.set_title(f'{label} histogram', fontsize=13)
            qq_ax.set_title(f'{label} Q-Q plot', fontsize=13)

        page_number = start // page_size + 1
        fig.suptitle(f'{title_prefix} residual normality diagnostics, page {page_number}', fontsize=18, y=1.01)
        plt.tight_layout()
        plt.show()
        plt.close()

    return summary


## Matched FC PCA Workflows


In [ ]:
pca_model_outputs = {}
pc_fc_column_results = []
pc_to_pc_results = []

for spec in PCA_MODEL_SPECS:
    print('\\n' + '=' * 90)
    print(spec['label'])
    print('=' * 90)
    print('FC covariates:', spec['fc_covariates'])

    cognitive_scores, cognitive_score_cols = load_cognitive_efa_scores(spec['model_name'])
    df = df.drop(columns=cognitive_score_cols, errors='ignore').merge(
        cognitive_scores,
        on='src_subject_id',
        how='left',
    )
    print(f'Loaded {len(cognitive_score_cols)} cognitive EFA factor columns for {spec["model_name"]}.')

    df, fc_resid_cols = residualize_fc_variant(
        df,
        raw_fc_cols,
        covariates=spec['fc_covariates'],
        output_suffix=spec['prefix'],
    )
    fc_residual_labels = [strip_fc_resid_suffix(col) for col in fc_resid_cols]
    fc_residual_normality = plot_residual_normality_diagnostics(
        df,
        fc_resid_cols,
        labels=fc_residual_labels,
        title_prefix=f"{spec['short_label']} FC residuals",
        page_size=8,
    )

    column_results = compute_corrected_correlations(
        df,
        cognitive_score_cols,
        fc_resid_cols,
        model_name=spec['model_name'],
        correlation_family='cognitive_efa_x_fc_column',
        required_nonmissing=[SES_VARIABLE] if spec['prefix'] == 'ses' else None,
    )
    pc_fc_column_results.append(column_results)
    display_correlation_summary(
        column_results,
        title=f"{spec['label']}: cognitive EFA factor x FC column correlations",
    )

    fc_result = run_fc_pca_with_bootstrap(
        df,
        fc_resid_cols,
        analysis_label=f"{spec['short_label']} FC PCA",
        score_prefix=f"{spec['prefix']}_fc",
        n_boot=N_FC_BOOTSTRAPS,
    )
    df = attach_score_columns(df, fc_result['scores'])


    # Export the fitted FC-PCA coordinates so downstream clustering notebooks
    # visualize the established PCA solution rather than fitting another PCA.
    fc_score_export = df[['src_subject_id', *fc_result['score_cols']]].copy()
    fc_score_path = PCA_RESULTS_DIR / f'pca_fc_scores_{spec["model_name"]}.csv'
    fc_score_export.to_csv(fc_score_path, index=False)

    fc_variance_export = pd.DataFrame({
        'component': np.arange(1, len(fc_result['var_exp']) + 1),
        'explained_variance_ratio': fc_result['var_exp'],
    })
    fc_variance_path = PCA_RESULTS_DIR / f'pca_fc_variance_{spec["model_name"]}.csv'
    fc_variance_export.to_csv(fc_variance_path, index=False)

    print('Saved FC-PCA scores to:', fc_score_path)
    print('Saved FC-PCA variance to:', fc_variance_path)

    pc_results = compute_corrected_correlations(
        df,
        cognitive_score_cols,
        fc_result['score_cols'],
        model_name=spec['model_name'],
        correlation_family='cognitive_efa_x_fc_pc',
        required_nonmissing=[SES_VARIABLE] if spec['prefix'] == 'ses' else None,
    )
    pc_results['fc_pc'] = pc_results['target']
    pc_to_pc_results.append(pc_results)
    display_correlation_summary(
        pc_results,
        title=f"{spec['label']}: cognitive EFA factor x FC PC correlations",
    )

    pca_model_outputs[spec['model_name']] = {
        'spec': spec,
        'cognitive_score_cols': cognitive_score_cols,
        'fc_resid_cols': fc_resid_cols,
        'fc_residual_normality': fc_residual_normality,
        'fc': fc_result,
        'pc_fc_column_results': column_results,
        'pc_to_pc_results': pc_results,
    }

no_ses_model = pca_model_outputs['without_ses_residualization']
ses_model = pca_model_outputs['with_ses_residualization']
no_ses_pc_fc_results_df = no_ses_model['pc_fc_column_results']
ses_pc_fc_results_df = ses_model['pc_fc_column_results']
no_ses_pc_to_pc_results_df = no_ses_model['pc_to_pc_results']
ses_pc_to_pc_results_df = ses_model['pc_to_pc_results']


## Relationships Between pMTG FC and SES

Test Pearson correlations between income-to-needs ratio (INR) and each pMTG-network FC measure. These tests use FC residualized for the standard covariates (age, sex, study site, handedness, and mean framewise displacement) without residualizing FC for INR. False discovery rate correction is applied across the full FC-SES test family.


In [ ]:
# Use FC residuals from the model that controls for the standard covariates but not INR.
fc_ses_columns = no_ses_model['fc_resid_cols']

# Correlate observed INR with every left/right pMTG-to-network FC measure.
fc_ses_results_df = compute_correlations(
    df,
    measures=[SES_VARIABLE],
    fc_columns=fc_ses_columns,
)

# Correct the complete FC-SES family for multiple comparisons.
fc_ses_results_df = apply_multiple_comparison_corrections(
    fc_ses_results_df,
    alpha=ALPHA,
)
fc_ses_results_df = fc_ses_results_df.rename(
    columns={'measure': 'ses_measure', 'col': 'fc_measure'}
)
fc_ses_results_df['abs_r'] = fc_ses_results_df['r'].abs()
fc_ses_results_df['fc_residualization'] = 'standard_covariates_without_inr'
fc_ses_results_df['n_tests_in_fdr_family'] = len(fc_ses_results_df)
fc_ses_results_df = fc_ses_results_df.sort_values(
    ['p_fdr', 'p', 'abs_r'],
    ascending=[True, True, False],
).reset_index(drop=True)

print(f'FC-SES tests: {len(fc_ses_results_df)}')
print(
    'FC-SES sample-size range: '
    f'{fc_ses_results_df["n"].min()}-{fc_ses_results_df["n"].max()}'
)
print('FDR-significant FC-SES correlations:')
fc_ses_fdr_results_df = fc_ses_results_df.loc[
    fc_ses_results_df['sig_fdr']
].copy()
if fc_ses_fdr_results_df.empty:
    print('None')
else:
    display(fc_ses_fdr_results_df)

print('Top 20 FC-SES correlations by absolute r-value:')
display(
    fc_ses_results_df.sort_values('abs_r', ascending=False).head(20)[
        ['ses_measure', 'fc_measure', 'r', 'p', 'p_fdr', 'p_bonf', 'n', 'abs_r']
    ]
)

# Save all corrected FC-SES results, not only significant associations.
fc_ses_results_path = PCA_RESULTS_DIR / 'fc_ses_correlations_without_inr_residualization.csv'
fc_ses_results_df.to_csv(fc_ses_results_path, index=False)
print('Saved FC-SES correlations to:', fc_ses_results_path)


## Combined FC Association Tables


In [ ]:
pc_fc_column_results_df = pd.concat(pc_fc_column_results, ignore_index=True)
pc_to_pc_results_df = pd.concat(pc_to_pc_results, ignore_index=True)
all_pca_correlation_results_df = pd.concat(
    [pc_fc_column_results_df, pc_to_pc_results_df],
    ignore_index=True,
)

summary_cols = ['ses_model', 'correlation_family']
pca_correlation_summary = all_pca_correlation_results_df.groupby(summary_cols).agg(
    n_tests=('r', 'size'),
    n_sig_bonf=('sig_bonf', 'sum'),
    n_sig_fdr=('sig_fdr', 'sum'),
    min_p=('p', 'min'),
    min_p_fdr=('p_fdr', 'min'),
    max_abs_r=('abs_r', 'max'),
).reset_index()

print('FC PCA EFA-factor association summary:')
display(pca_correlation_summary)

for (model_name, family), family_results in all_pca_correlation_results_df.groupby(summary_cols, sort=False):
    print('\\n' + '=' * 90)
    print(f'{model_name}: {family}')
    print('=' * 90)

    fdr = family_results[family_results['sig_fdr']].copy()
    print('FDR-significant results:')
    if fdr.empty:
        print('None')
    else:
        display(fdr.sort_values(['p_fdr', 'p']))

    print('\\nTop 20 by absolute r-value:')
    display(
        family_results.sort_values('abs_r', ascending=False).head(20)[
            ['cognitive_efa_factor', 'target', 'r', 'p', 'p_fdr', 'p_bonf', 'n', 'abs_r']
        ]
    )
